In [2]:
import os

os.chdir("..")
os.chdir("..")
os.chdir("..")


In [3]:
!dir

 O volume na unidade G � Armazenamento
 O N�mero de S�rie do Volume � 2091-BE76

 Pasta de g:\py_projects\fast-english

23/07/2025  11:05    <DIR>          .
18/07/2025  07:49    <DIR>          ..
03/07/2025  16:47    <DIR>          .github
22/07/2025  22:58             4.617 .gitignore
03/07/2025  16:47                 9 .python-version
03/07/2025  16:53    <DIR>          .venv
22/07/2025  23:01    <DIR>          add_new_data
03/07/2025  21:20    <DIR>          app
12/07/2025  20:33    <DIR>          database
03/07/2025  16:47    <DIR>          docs
03/07/2025  16:47           456.432 poetry.lock
03/07/2025  16:47             1.231 pyproject.toml
03/07/2025  16:47               165 README.md
03/07/2025  16:47               203 requirements.txt
23/07/2025  23:20         1.038.672 set_difference.txt
03/07/2025  16:47    <DIR>          tests
03/07/2025  16:47    <DIR>          trash
               7 arquivo(s)      1.501.329 bytes
              10 pasta(s)   146.054.000.640 bytes dispon�

In [4]:
import tkinter as tk
from pathlib import Path
from app.toolkit.utils.data_loader import DataLoader
import re
from tkinter import simpledialog
import json
from tqdm import tqdm

class TextReaderApp(tk.Frame):
    def __init__(self, parent=None, **kwargs):
        super().__init__(parent, **kwargs)
        self.known_words = self.prep_data_match_word()

        scrollbar = tk.Scrollbar(self)
        scrollbar.pack(side="right", fill="y")

        self.text_widget = tk.Text(self, wrap="word", font=("Arial", 14), yscrollcommand=scrollbar.set)
        self.text_widget.pack(expand=True, fill="both")

        scrollbar.config(command=self.text_widget.yview)

        self.send_button = tk.Button(self, text="Enviar", command=self._process_text)
        self.send_button.pack(pady=(0, 5))

        self.send_button = tk.Button(self, text="Criar banco de dados com palavras", command=self._save_data_words)
        self.send_button.pack(pady=(0, 5))

        # self.btn_listar_conhecidas = tk.Button(self, text="Listar Palavras Conhecidas no Texto", command=self._listar_palavras_conhecidas)
        # self.btn_listar_conhecidas.pack(pady=(0, 5))


        self.btn_remover_conhecidas = tk.Button(self, text="Remover Palavras Conhecidas do Texto", command=self._remover_palavras_conhecidas_do_texto)
        self.btn_remover_conhecidas.pack(pady=(0, 5))

        self.btn_remover_desconhecidas = tk.Button(self, text="Remover Palavras Desconhecidas do Texto", command=self._remover_palavras_desconhecidas_do_texto)
        self.btn_remover_desconhecidas.pack(pady=(0, 5))

        self.btn_palavras_unicas = tk.Button(self, text="Manter Apenas Palavras Únicas", command=self._manter_palavras_unicas)
        self.btn_palavras_unicas.pack(pady=(0, 5))


        # Coluna do stats_label
        # self.stats_label = tk.Label(self, justify="left", font=("Arial", 12), anchor="nw")
        # self.stats_label.grid(row=0, column=1, padx=10, sticky="nw")  # Alinha no topo à esquerda

        self.stats_label = tk.Label(self, justify="left", font=("Arial", 12), anchor="nw")
        self.stats_label.pack(side="left", padx=10, anchor="n")

        self.list_data_words_in_text = []
        self._configure_tags()

    def prep_data_match_word(self):
        """
        Função otimizada que prepara os dados para a busca de palavras.
        """
        data_loader_words = DataLoader(base_path="database/extract_data_video/data/extracted_data/words/data_organize")
        data_loader_phrases = DataLoader(base_path="database/extract_data_video/data/extracted_data/phrases/data_organize")

        # Junta as listas diretamente
        lista_total = data_loader_words.get_all_words() + data_loader_phrases.get_all_words()

        # Cria os pares (palavra, caminho) com list comprehension
        set_data_words = [
            {"word": word, "path": Path(info["path"])}
            for info in tqdm(lista_total, desc="Processando palavras")
            for word in Path(info["path"]).stem.replace("_", " ").lower().split()
        ]

        return set_data_words

    def find_match_word(self, word, set_deta_words):
        """
        Função que procura por palavras que possuem a tag "tag" no nome.
        """
        word = word.lower()
        
        list_match_word = []
        for data_word in set_deta_words:
            word_name = data_word.get("word", "")

            if word == word_name:
                list_match_word.append(data_word)
        
        return list_match_word
        

    def _save_data_words(self):
        file_name = simpledialog.askstring("Salvar palavras", "Digite o nome do arquivo (sem extensão):")
        
        if not file_name:
            return  # Usuário cancelou

        MAX_POR_PALAVRA = 2
        contador_por_palavra = {}
        save_words = []

        # Novos requisitos: As palavras as tão repetindo pois estão vindo do mesmo caminho
        for dict_match_word in self.list_data_words_in_text:
            word = dict_match_word["word"].lower()
            new_path = str(dict_match_word["path"])

            if contador_por_palavra.get(word, 0) < MAX_POR_PALAVRA and new_path not in save_words:
                save_words.append(new_path)
                contador_por_palavra[word] = contador_por_palavra.get(word, 0) + 1

        save_path = Path("database/vocabulary/save_words") / f"{file_name}.json"
        save_path.parent.mkdir(parents=True, exist_ok=True)

        with open(save_path, 'w', encoding="utf-8") as json_file:
            json.dump(save_words, json_file, ensure_ascii=False, indent=4)

        print(f"Palavras salvas em: {save_path}")

    def _listar_palavras_conhecidas(self):
        # Cria janela popup
        popup = tk.Toplevel(self)
        popup.title("Palavras Conhecidas no Texto")

        listbox = tk.Listbox(popup, selectmode=tk.SINGLE, width=50, height=20)
        listbox.pack(padx=10, pady=10)

        # Usar set para evitar repetições
        self.palavras_conhecidas_unicas = sorted(set(
            data['word'] for data in self.list_data_words_in_text
        ))

        for word in self.palavras_conhecidas_unicas:
            listbox.insert(tk.END, word)

        btn_remover = tk.Button(popup, text="Remover Palavra Selecionada", command=lambda: self._remover_palavra_base(listbox))
        btn_remover.pack(pady=(0, 10))

    def _remover_palavra_base(self, listbox):
        selection = listbox.curselection()
        if not selection:
            return

        index = selection[0]
        palavra = self.palavras_conhecidas_unicas[index]

        # Remove todas as ocorrências dessa palavra na base
        self.known_words = [w for w in self.known_words if w["word"] != palavra]

        # Atualiza a listbox
        listbox.delete(index)

        print(f"Palavra removida da base: {palavra}")

    def _manter_palavras_unicas(self):
        text = self.text_widget.get("1.0", tk.END)
        seen = set()
        result = []

        for match in re.finditer(r"[a-zA-Z]+", text):
            word = match.group()
            key = word.lower()

            if key not in seen:
                seen.add(key)
                result.append(word)

        texto_resultante = " ".join(result)
        self.text_widget.delete("1.0", tk.END)
        self.text_widget.insert("1.0", texto_resultante)

    def _remover_palavras_conhecidas_do_texto(self):
        texto = self.text_widget.get("1.0", tk.END)
        palavras_para_remover = []

        for match in re.finditer(r"[a-zA-Z]+", texto):
            palavra = match.group()
            if self.find_match_word(palavra, self.known_words):
                palavras_para_remover.append(palavra)

        for palavra in set(palavras_para_remover):
            # Remove apenas a palavra completa
            texto = re.sub(rf'\b{re.escape(palavra)}\b', '', texto)

        # Limpa espaços extras e finaliza
        texto = re.sub(r'\s+', ' ', texto).strip()

        self.text_widget.delete("1.0", tk.END)
        self.text_widget.insert("1.0", texto)
        self._process_text()


    def _remover_palavras_desconhecidas_do_texto(self):
        texto = self.text_widget.get("1.0", tk.END)
        palavras_para_remover = []

        for match in re.finditer(r"[a-zA-Z]+", texto):
            palavra = match.group()
            if not self.find_match_word(palavra, self.known_words):
                palavras_para_remover.append(palavra)

        for palavra in set(palavras_para_remover):
            # Remove somente a palavra isolada (com limites de palavra)
            texto = re.sub(rf'\b{re.escape(palavra)}\b', '', texto)

        # Remove espaços duplos e linhas em branco
        texto = re.sub(r'\s+', ' ', texto).strip()

        self.text_widget.delete("1.0", tk.END)
        self.text_widget.insert("1.0", texto)
        self._process_text()

    def _configure_tags(self):
        self.text_widget.tag_config("green", foreground="green")
        self.text_widget.tag_config("gray", foreground="gray")

    def _process_text(self):
        self._clear_tags()
        text = self.text_widget.get("1.0", tk.END)
        self.total_words = self.known_words_count = 0
        self.list_data_words_in_text = []

        for match in re.finditer(r"[a-zA-Z]+", text):
            word = match.group()
            start, end = match.start(), match.end()
            self.total_words += 1

            match_data = self.find_match_word(word, self.known_words)
            if match_data:
                self.known_words_count += 1
                self.list_data_words_in_text.extend(match_data)
                tag = "green"
            else:
                tag = "gray"

            self.text_widget.tag_add(tag, f"1.0+{start}c", f"1.0+{end}c")

        self._show_statistics()

    def _clear_tags(self):
        self.text_widget.tag_remove("green", "1.0", tk.END)
        self.text_widget.tag_remove("gray", "1.0", tk.END)

    def _show_statistics(self):
        total = self.total_words
        known = self.known_words_count
        unknown = total - known
        percent = (known / total * 100) if total else 0

        stats_text = (
            f"Total de palavras: {total}\n"
            f"Conhecidas: {known}\n"
            f"Desconhecidas: {unknown}\n"
            f"Compreensão: {percent:.1f}%"
        )
        self.stats_label.config(text=stats_text)


if __name__ == "__main__":
    try:
        root = tk.Tk()
        root.title("Leitura de Texto")
        app = TextReaderApp(root)
        app.pack(expand=True, fill="both")

        # # Texto inicial (opcional)
        # example_text = """How many-teste pieces you retrieve from your RAG system affects the result."""
        # app.text_widget.insert("1.0", example_text)
        root.mainloop()
    except Exception as e:
        print(f"Erro: {e}")
    finally:
        root.destroy()



Processando palavras: 100%|██████████| 10827/10827 [00:00<00:00, 27891.41it/s]


Palavras salvas em: database\vocabulary\save_words\how_to_have_better_finances_than_95%_of_people.json


TclError: can't invoke "destroy" command: application has been destroyed

In [ ]:
end

In [ ]:
What if I told you that having better finances than 95% of people is actually very easy. You don't need a six figure salary. You definitely don't need to haggle over a $5 coffee and you don't need to cancel everything fun in your life. I've helped tons of people go from feeling like they are living paycheck to paycheck to saving their first $10,000, building six figure investments and not stressing out over money. And in this video, I'm going to show you exactly how to do that. So you can get ahead of 95% of people in just three months. Step one, master the numbers that 95% of people ignore. Most people think they have a handle on their money, but ask them one simple question. How much do you spend in total every month and suddenly blank stares? That's exactly why so many people feel stuck. If you don't know your monthly spending, you are basically flying blind. You're making random financial decisions and praying it all works out. But the truth is it won't. If you apply at least half of what I tell you in this video, you'll be miles ahead of everyone else. Think about it. Half the people I speak to don't even know their own annual income. 90% of people do not know their total debt. ninety five% of people don't even know when their debt will be paid off. And almost nobody calculates their crossover point. More on that later. It's going to change the game for you. See a lot of people will go their entire lives without knowing these basic numbers and they will feel behind because of it. They might actually earn a lot of money. They might actually be in a pretty good financial place, but they feel horrible about their money. You don't have to feel that way. Once you take control of your numbers, you're going to feel good about your money. No more feeling overwhelmed. No more wondering where's all the money going. You're going to make moves that actually build wealth and you're going to see it happening in front of your eyes. So how do you do it? How do you go from financially lost to completely in control? It starts with asking yourself a few brutally honest questions. And I'm going to ask you these right now to figure out your five key numbers. Number one, what's your burn rate? You know, when I ask people how much they spend every month, they pick a number, which you can tell is a complete lie. First of all, it's always a round number. Oh, three thousand. How convenient that the number ended in a zero. Shocking. And second, they answer with a feeling. I feel like it's too much. I didn't ask for a feeling. I asked for a number. Then I actually have them check their bank and credit card statements. And oftentimes it's like 4,500. Whatever number they thought, it's way more. Now that number is their burn rate. That's the total amount of money they are spending every month. If you don't know your burn rate or how much you're spending every month, it's very difficult to make any type of plan. You're sort of just going along on the right of life. Oh, please don't take me off a cliff. I hope I make it. We don't want that. Okay. Want to be calm, cool, methodical. So here's how to figure out your burn rate. By the way, no spreadsheets, no budgeting apps. We're just going to do it right now. Log into your bank account and your credit card accounts. Look at the last three months of your spending. Add up the total amount spent each month. Yes, I said total. Oh, Ramit, does that include the water that comes out of my sprinklers? Yes. And Ramit, does that include the gas I had to fill, but also that one time I had to fill extra gas because I almost got it. I said total. What is not clear about the word total? T-O-T-A-L, total, everything. Then divide by three. That's your average monthly spending. Okay. I know, Ramit, that didn't include my December trip. It didn't include... We'll get there. Okay. Right now, we just need to pick a number so you have a baseline to work from. That's your burn rate, your real number, know it, own it. If anything, it's probably a little higher. So once you just add fifteen percent on top. Okay. If you don't like what you see, don't panic. Also don't kill the messenger. Okay. I'm not the bad guy here. I'm trying to help you figure out what's going on in your financial life. In the next steps, I'm going to show you exactly how to fix this number. Number two, how big is your money black hole? Listen, the second you see your actual debt in black and white, everything shifts. You don't have to guess anymore. You can actually take control. Do you know you can live a rich life even if you're in debt? First off, we need to measure how much debt we're in. Let's measure the size of our money black hole. This is what you do. List off everything you owe. All of your debt, credit cards, student loans, car loans, mortgage, personal loans, that God forsaken Kohl's card you opened up so you could get $10 off that substandard pair of jeans. Write that down too. Next, write down the interest rates or the APR and the minimum monthly payments for each. Do the math. Reason to a debt payoff calculator. If you only pay the minimums, how long until you're debt free? You can also use our free debt calculator, which I'll link below. Well, let me just put it this way. If you have credit card debt and you're paying 27% interest, you're probably going to be paying it off for decades if you pay the minimum. But once you see the real numbers, that's actually magic. You can finally take control. This is kind of like somebody avoiding seeing the doctor for a decade. They're so afraid, "Oh my God, what's wrong?" Maybe the doctor gives you a little bit of bad news. "Hey, you got to lower your cholesterol." Whatever. "Okay, cool. Now I know what to do. At least I can go to sleep at night knowing I have a plan." For example, if you add just $100 extra per month towards your high interest credit card debt, you can actually shave years off of that debt payoff. I love it. A money black hole only gets bigger when you ignore it. But once you face it, you can defeat it. Number three, are you walking on a tightrope? I know a lot of people try to save. But when I ask, "How much are you actually saving every month?" "Well, I really tried to, but I was bad last month. I know I was bad. I went to Arby's. I know I shouldn't have done that, but the horseradish sauce just called to me. I know I should save. I'll try to be good next month. Good." Putting aside the extremely weird moral valence that Americans love to assign to money, and also the fact that you have an awful palate if you really chose to eat at Arby's, why are you talking about trying to save money? I don't try to brush my teeth every night. I just do it. And saving money is actually easier than either of those things because you can set it up to happen automatically. Saving is not just some vague good habit. It actually can be the difference between balance and disaster when life throws you a curveball. Now listen, all jokes aside. In times of financial chaos, you want to have a savings account because if you get laid off or your partner, if something happens to an elderly parent or kid, you need to be able to tap into your savings and use it to live your life. You do not want your back against the financial wall. It causes you to have to make terrible decisions. When people don't have savings, they stop taking medicine. They die. Really bad things happen. That is why you should be saving at least 5 to 10% of your take home pay every month and you should make it happen automatically. Now if you're not saving anything right now, don't feel guilty. Let's start right now with 20 bucks. That's less than the price of takeout and you can take that dollar amount or any dollar amount, it doesn't really matter, and you can prove to yourself that I am the kind of person who saves. Again, you can use the material in happening every single month, you're not even going to miss the money, then you can increase that number. You'll go from walking a financial tightrope to building a bridge with guardrails and lights. It feels safe. Number four, is your money working as hard as you are? I got to tell you, I have no interest in working harder and harder in my life. It's like playing a game of Mario where it just gets harder every single level you get to. Fun for a video game, not so fun for life. I'm trying to chill. I work hard, great. I worked hard when I was in my 20s and I like a nice little sweater once in a while. I want to take a vacation. Well, guess what? Wealthy people don't just earn money, they put it to work and that is the power of investing. I'm not talking about some freaking PE ratio screaming buy, sell guy sweating through his poorly fitting suit. I'm talking about smart investors who actually sit still. Smart investors are doing way less work than you think. They're not sitting around picking stocks. They're not glued to CNBC. They start early, they stay consistent and investing is boring. They set it up to happen automatically and they let time do the heavy lifting. So here's the golden rule. I want you to aim to invest at least 10% of your take home pay. If you can't hit that yet, okay, start small. Let me show you some numbers. Even a hundred dollars a month with a modest 7% return becomes almost a quarter million dollars after 40 years. I know I'm about to get 6,500 comments saying, "Oh, $250,000 in 40 years won't be anything." I've already included inflation in my calculations. I always do. That's the real return, not the nominal return. So please stop trying to nitpick calculations as a way to avoid taking action. If you're still waiting for the perfect time to invest, bad news. You leaving a snarky, also wrong comment on YouTube is not going to help. Good news, now is a great time to start investing. Remember time in the market beats timing the market. So get started right now. Your future self will thank you. Number five, are you in the housing red zone? You wouldn't believe how many people I talk to who are obsessed with how expensive bread is at the grocery store or how their husband spends too much on energy drinks. But I take a look at all their numbers and the energy drinks and the eggs have nothing to do with it. They are spending too much on housing and it is just that simple. If you are spending more than 28% of your gross income on housing, congratulations. You are officially in the housing red zone and you can join millions of other people who are there as well. Now let me show you how to calculate this number right now. Then we'll talk about what to do about it. Take your monthly rent or mortgage payment and your utilities and any other expenses for housing, repairs, even the gas you spend driving on the weekend to go to Home Depot. Take that number, divide it by your monthly gross salary. That's before taxes and deductions and then multiply it by a hundred. Now it used to be that we would try to aim to get below 28%. These days it's pretty hard to stay below that number, especially in expensive cost of living cities, especially if you're just starting. I hate that this is the case. That's why I'm a huge yimby, somebody who wants to build more and more housing so that housing prices come down. But let's talk about reality. Let's talk about today. If you're at 29%, okay, you could probably get by. It's not going to be a disaster. Maybe you're at 32% starts to get a little risky. 34%. If you have 34% of money spent on housing, I can already tell what's going on in your household. You're stressed about money. If you're in a relationship, you're fighting about money and you're fighting about random expenses. Why'd you buy those M&Ms? It's not about the M&Ms. If you look up, you're spending 35% of gross on your housing. That is why you are stressed out. That is why you can't save. That is why everything feels so tight. Now again, it's not easy to stay below 28%. Housing is historically expensive for political reasons, but you need to know where you are, first of all, because if you're over it, okay, then the question is, what do I do? Maybe I cut over here. Maybe I pay off debt more aggressively. You also can start exploring other options. Do I downsize? If I'm single, do I get a roommate? Am I renting out a room on Airbnb? Am I moving somewhere slightly cheaper? The point is you need to first know how much higher than 28% you are, and then you need to make a plan. Because if housing is devouring your income, getting ahead will feel like running a marathon while wearing a backpack full of bricks. Your housing should be a launch pad for your rich life. It should not be a place for you to be house poor. Now look at all the numbers you just calculated. Write them down. If you don't see them in black and white, they're just these vague things, feelings floating around, but we need to actually see the numbers and combine those with how we feel about money. Okay, great. You wrote them down. Now that you've got full clarity on your financial situation, one other thing. Knowledge alone does not build wealth. Okay? So what's next? You're going to build a system that makes your money grow on autopilot. What you're probably noticing is that in order to live a rich life, you have to question a lot of the common things that you were taught about money. Let's talk about something else that almost nobody questions, their financial advisor. Do you know how much you pay your financial advisor or how much your parents pay theirs? Most people actually have no idea. Most people pay a financial advisor a percentage, something like 1% of assets under management. Doesn't sound like a lot. 1%, big deal. That 1% can cost you hundreds of thousands of dollars over your lifetime. And the truth is most people can probably DIY their own money themselves. But if you have a complex situation or a large portfolio, or if you just have specific scenarios you want guidance on. I'm all for hiring a professional, financial advisor or financial planner. Okay, now that your money's starting to work for you, let's put the whole thing on autopilot. Step two, automate like the top 5%. Imagine waking up, checking your bank account, seeing that your bills are paid, your savings are stacking up and you have the money left to spend guilt-free on whatever you want. No stress, no spreadsheets, no, "Oh my God, did I forget to pay my credit card bill again?" Only one quick thing, you don't even need to log into your account every freaking day, okay? I don't. Your money's automatically set, it goes where it needs to go and you can spend on the things you love without having to check in every day. "Oh, Bank of America, why am I staying with this sh** bank? Do I have 15 cents?" No, you already know and also you switched off a B of A. So how do you get there? Well, let me tell you where most people mess up. They save whatever's left at the end of the month and truthfully, that's the problem right there. That's why they never get ahead because if you wait until the end of the month to save, there's not going to be any money left. I can tell you right now, the top 5% do the opposite. They pay themselves first. Let me show you how this works for you. Before you even get paid, a portion of your paycheck goes straight to your 401k. On the fifth of the month, automatic transfers send money to your savings and investment account. A portion of that money goes into a high yield savings account, roughly 5% to 10% for your emergency fund, upcoming big purchases, maybe a down payment on a house. Another 5% to 10% goes straight into a Roth IRA where it grows tax free. That money turns into big money over the long term. On the seventh, your system automatically pays all your bills, rent, utilities, insurance, so you're never missing a payment or incurring late fees. By the way, your credit card is also paid off in full on this date. You're not carrying a balance, no interest payments, no more giving the bank or credit card your hard earned money. Whatever's left, spend it however you want. Guilt free. Now, if you don't want to track every dollar, great. You don't have to. You just use the system I outlined. Once your system is running, every dollar is working in the background to help you live your rich life. No need to compare the price of lettuce. No need to feel guilty. I'm a moral failure because I actually spent 13 extra cents on getting oat milk. Doesn't matter. Your money's going where it needs to go by percentage. It's all happening automatically. Once that's happening, you can focus on something even bigger, your crossover point. Step three, calculate your crossover point. Imagine waking up one day and realizing you never have to work again. That is your crossover point. That's the moment your investments generate enough income to fully cover your expenses automatically. Guys, you don't need to win the lottery in order for this to happen. You don't need to be making a million dollars a year. You can build a system that makes this possible and helps you achieve financial independence. Once you have the crossover, you can keep working if you want to, or maybe you decide you want to work part-time, take a sabbatical, go out to brunch for three hours a day, take your dog for dog yoga. It's up to you. That's freedom. Let's calculate the math. I'll break down how it actually works for you right now. First, figure out how much money you need to actually cover your monthly expenses. That's your target number. That's the amount you need to live comfortably without a paycheck. Next, plug that number into a retirement calculator to see how much you need to invest to generate that amount every month. I'm going to link a calculator below. That is your crossover number. From here, you can actually play with the variables. You want to cut expenses? Cool. You'll need to save less. You want to earn more? You can save and invest even faster. Do both. You can actually hit your crossover point a lot faster and only minimally have to cut back on your lifestyle. Let me show you how even small changes can massively shift your timeline. Let's say you make eighty thousand a year, you spend $6,000 a month, and you invest 10% of your income. At that pace, with a conservative 7% return on your investments, you'd reach financial freedom in about 42 years. Okay, not bad. That's the default. Now, let's speed things up. Cut your expenses to $3,000 a month. Boom. Freedom in under 13 years. You're like, "Rameen, I'm not trying to live like that." Okay, no problem. Let's say you increase your income by 30% and you invest all of it. That alone could cut your timeline to 24 years. If you both increase your income by 30% and cut your expenses by just 30%, not 50, just 30, you could reach financial freedom in just 14 years. The takeaway here is you don't need to just live on rice and beans or make $800,000 a year. You do need to be intentional about earning more, spending less, or ideally both. Now, how do you get there? Let me show you. Now, this is where 95% of people get stuck. They don't know how to invest. But I got to say something. People will literally leave comments in my YouTube videos going, "How do I invest? You talk about investing, but you never of aggravation, "But how do you invest?" That's like me saying, "But where do I find water in the developed world?" Go and turn on a tap. If you don't know where the tap is, go on YouTube and say, "Where do I drink water in a big city?" It's everywhere to be found around you, but you have to go and find it. Nobody's going to come and say, "Hello, little Benjamin. Let me sit you down and teach you all about the difference between diversification and asset allocation. F***ing take some responsibility." All right, for the 3% of people who are not totally turned off by me and this video and are actually still watching and learning how to invest, I'm going to f***ing show you how to invest. People think investing means stock picking, watching charts, listening to some guy, Chris Crone on TikTok telling you some other scammy idea he has. No, that's not what investing looks like. When my family asked me what to invest, I talk about very simple investments. Index funds, target date funds. Let's break it down. Index funds are basically low cost bundles of stocks that track the market. Historically, they've returned about 7% per year after inflation. That basically means you have a very inexpensive way of buying investments. It's really simple and believe it or not, you can actually outperform most expensive money managers. Your money grows in the background. If you want to make it even easier, you can use a target date fund. With a target date fund, you just pick the year you want to retire and you invest in the fund with that date and it adjusts, becomes more conservative as you get older. There's great target date funds at Vanguard, Schwab or Fidelity and that is how you start to invest. Now, why this actually matters, guess what? Not everybody wants to retire early. That's okay. The crossover point isn't just about quitting your job. It's about giving you options. It's about waking up and knowing if you don't love your job anymore, maybe you don't have to stay there. If you don't want to work 50 hours a week, maybe you can talk about going part time. If you want to travel a lot more, you know you have the money to do it. This is using money for freedom, flexibility and having the power to design your life on your terms. So, ask yourself, do I want to reach my crossover point? And if yes, what's my path to get there? Because you get to decide how fast you want to go and how you get there and what your life looks like once you do it. Which brings me to step four, scale up your new rich life. Let's zoom out for a second. Once you've calculated your crossover point, everything really starts to change. You might not be financially free yet, but you are on the path. You've moved from this nebulous concept of someday to something real. You know the number, you know what it takes. That alone puts you ahead of 95% of people. And whether you're one year away or 10 years, now is the time to start thinking like someone who's already financially free. Because if you wait until you hit your number someday, what a terrible way to go through life. People worry about their money forever, then maybe they reach financial freedom, but they never even enjoyed the process and now they don't even know what to do. That's not going to be you. You didn't come this far just to stare your net worth like it's a score in a video game. A lot of people see money as a limitation. It stresses them out. It keeps them trapped. But the people who are watching this, the 5%, they know money is a tool. It's a lever. It's a passport to be able to travel to the kind of life you want. So here are a few questions to help you design your new rich life. What financial rules do I follow and which ones am I going to throw out? Remember, you don't have to do what everyone else does. Part of creating your rich life is being unapologetically different. For example, maybe you love renting because you don't want to fix your leaky roof, or you don't care about cars so you drive an old modest sedan, or you prioritize travel, or you spend a ton of money on private school for your kids. It's up to you. Your money, your values, your rules. Where am I spending too little? Now that you're financially free or on the path, what is something you love but you've been holding back on? If it's health, maybe it's time to get a therapist for mental health or attend yoga classes. If it's convenience, maybe you hire someone to help out around the house every once in a while. If it's experiences, maybe it's time to take that cool camping trip or splurge on that international trip. This is the moment you start to enjoy the rewards of your hard work. How do I give back? Mental wealth isn't just about having more money in your bank account. It's about what you do with it. That might mean helping your parents retire, funding a passion project, or supporting a cause you care about in real life, not just on Instagram. This is your opportunity to use money to create meaning. What's my next chapter? Now that money is not your primary constraint, what do you actually want to do? What dreams did you use to have that you left behind or put on pause because of financial limitations? Maybe you always wanted to learn piano or travel the world. Maybe you wanted to go and eat different kinds of food and not look at the price on the menu. Or maybe you want to finally start that business that's been sitting in the back of your head for years. This is your chance to design your rich life on your terms, no limits. Now's the time. Your rich life is yours to design. If you are serious about making this real, your first major milestone is hitting 100K. Why 100K? Because that's when everything changes. Your money starts working harder than you do and you finally feel ahead. If you want the roadmap to get there, I break it all down right here. 


So how does this kind of our model real on that the rest us use them for what being is can be quite And if humanity last exam happened in season League want to take either frankly you have your knowledge me even more interesting should start conversation not end Who has right judge whether or AI was correct big question Music welcome back Super Data Science doing today John Thank having yet again excited here as always Yes we were just starting recording many times been show than knew came into green room like before get studio said going time but it sixth one first ever Episode January Exactly Old enough drink finally then about year later couple years at another after February my when took over from Open Conference East met up quickly set camera live person lots number six best think too do Okay currently writing th book sounds there working two now books better often rock bands know they make their album Bush band popular Oh Stone guess well funny People Machine Head Are some favorite everyone every track huge global hit world tours record company need next man five making seems happen all with never same throwing shade out No very much go necessarily something different conscious wrote four ago really new say else aim building upon second third edition most recent guide only gotten almost point zero deep learning an introduction thing meant someone diving engineer understand gets because pretty Nice Tell Sure had copy holding earlier organized few level language mean brought part difference between these evaluate philosophy parts usually speaking let phone build things scratch those three around evaluation example will talk cool played got yesterday late night tried faster cheaper above bigger months High where could longer believe larger terms makes sense used busy day far tough stay date also why lot change speaks way everything find comfort maybe keep air secret love title known give least applied read dive top seen summarize its specific wide spectrum ground running both Python code addition ton teaching platform which brings brand write comes work great technical publisher hands information expert little put order journey watching reading trying watch education hard especially field helpful long proposal mind son probably delighted introduce matter switched industry help massage Sand com subscribe through employer university teach month once week wild July course might founder original went him see honestly bet made pieces originally come idea architecture sky chat down shocked invented since anything hold still pack history relative space Brain nature papers leading would automatically true works large responsible gym whole words reasons including human feedback planet combination already amazing together simply lesson learn must rally missed tomorrow publication check future other courses coming rag class weeks fun crowd agent questions own days yours text worry called intelligent started friend mine student he partner Capital asked lunch drinks meet bring talking products by latest generation deliver while combine chips achieve single Purpose offer price performance across cutting edge power transform familiar advise Ash General independently invited bit financial expected looking technology become side rather thinking may call paying goal tackle problem dead replace name look promising whatever ask case such role enjoy constantly calling wait schedule meeting fish investment cycle plus market share happy walk business done angel investor remember recommend getting sooner Dog Patch location lives apartment wear gear life sign safe hope win thick accident follow main topic cover related gave current instead tend lead chase teams test quote fine good approach task decide individual organization marketing Therefore remind everybody against necessary without stuck web shadow doubt training lecture remove items similar method search match found Llama miss standard checks common hand bad clearly worse art double prove train system perfect claim saw did tests using law score deal word telling truth trust similarly stopped contents released within pulled off shelf reward signal checking any please eat issue seem problems solve agree war context concerned fair counter answer knows internet care less process finding fill points create behind generally academic extremely consumer themselves page table each row column theirs circle However small print shot saying added chain thought story technique opinion recommended integrated scale Imagine able colleague waiting complex plain English smarter collaborate easily cut half visit software goes multiple choice sentence free entirely allowed forget sometimes prefer naturally salt Switch middle temperature listening environment relatively turn increase opposite diverse playground students decision trouble hot water fits correctly consistently yourself smart thanks learned included useful enterprise hallucination rates cheap simple static expecting place potentially taking situation convincing describing stuff stick watermelon seeds reference piece theory anyone advice grow stomach sit Magic School Bus famous hear rate surprised particularly research beginning near resolution prevent online solution impossible hidden charge arena run professor his pit pick thoughts reason bringing somebody grade fan blind sick yellow stable whoever scenes route closely compare painted important total separate notice expectations factory united drive speed adoption accelerate design wanted am recommendation checked key application stand quietly production admit efficient annoying team higher job tune internal public treat sales figure close sale breed culture Stripe recently engine willing fraud sell document classical whose paper experiment degree catch ours easy sound step short term won early possible fall trap success past previous worked seven billion natural file present bunch calls certainly object slow expensive client results Laura L algebra takes math mathematics beautiful efficiently money add extra catastrophic collapse quality fast intensive comparative gone effective sorry hate garbage forced classes Effectively references structure board matches highest reliable further drop try punishment itself glad stamp signals metric among red Mars calculate Jupiter Venus threshold textbook value dependent earth blue marble low unfortunately promise clear act somehow conclusion along explanation respect audio image video several hours documents lava project answering differently perform update Critic territory prophecy tour gather website break final agency tool steps begin cliff immediately selection study victim action pass fail window plateau ceiling result hundred kill machines capable lie ahead jumping thoughtful possibility generous kidding meantime report Snake Oil pleasure workshop direct gap living modern hairdresser appointment Fantastic author highly soon hearing America guest surprise dedicated except shows round artificial network news weekly film major anchor sports weather laugh nobody office crazy listen fake ads toilet forward note social guy grew tired articles lately covered suffer publicly deny guarantee implement reflect needs manager editor writer excellent grateful support interested Otherwise review shorts continue Until


In [ ]:
end

In [ ]:
from pathlib import Path
from tqdm import tqdm  # se quiser ver progresso
import time


def prep_data_match_word():
    """
    Função otimizada que prepara os dados para a busca de palavras.
    """
    data_loader_words = DataLoader(base_path="database/extract_data_video/data/extracted_data/words/data_organize")
    data_loader_phrases = DataLoader(base_path="database/extract_data_video/data/extracted_data/phrases/data_organize")

    # Junta as listas diretamente
    lista_total = data_loader_words.get_all_words() + data_loader_phrases.get_all_words()

    # Cria os pares (palavra, caminho) com list comprehension
    set_data_words = [
        {"word": word, "path": Path(info["path"])}
        for info in tqdm(lista_total, desc="Processando palavras")
        for word in Path(info["path"]).stem.replace("_", " ").lower().split()
    ]

    return set_data_words


set_data_words = prep_data_match_word()

Processando palavras: 100%|██████████| 10829/10829 [00:00<00:00, 39182.47it/s]


In [ ]:
match_word

[{'word': 'aberrant',
  'path': WindowsPath('database/extract_data_video/data/extracted_data/words/data_organize/adjetivos/descição_de_arte/aberrant')},
 {'word': 'admirable',
  'path': WindowsPath('database/extract_data_video/data/extracted_data/words/data_organize/adjetivos/descição_de_arte/admirable')},
 {'word': 'amazing',
  'path': WindowsPath('database/extract_data_video/data/extracted_data/words/data_organize/adjetivos/descição_de_arte/amazing')},
 {'word': 'antique',
  'path': WindowsPath('database/extract_data_video/data/extracted_data/words/data_organize/adjetivos/descição_de_arte/antique')},
 {'word': 'artistic',
  'path': WindowsPath('database/extract_data_video/data/extracted_data/words/data_organize/adjetivos/descição_de_arte/artistic')},
 {'word': 'brief',
  'path': WindowsPath('database/extract_data_video/data/extracted_data/words/data_organize/adjetivos/descição_de_arte/brief')},
 {'word': 'classical',
  'path': WindowsPath('database/extract_data_video/data/extracted_d

In [ ]:
set_data_words

[{'word': 'aberrant',
  'path': WindowsPath('database/extract_data_video/data/extracted_data/words/data_organize/adjetivos/descição_de_arte/aberrant')},
 {'word': 'admirable',
  'path': WindowsPath('database/extract_data_video/data/extracted_data/words/data_organize/adjetivos/descição_de_arte/admirable')},
 {'word': 'amazing',
  'path': WindowsPath('database/extract_data_video/data/extracted_data/words/data_organize/adjetivos/descição_de_arte/amazing')},
 {'word': 'antique',
  'path': WindowsPath('database/extract_data_video/data/extracted_data/words/data_organize/adjetivos/descição_de_arte/antique')},
 {'word': 'artistic',
  'path': WindowsPath('database/extract_data_video/data/extracted_data/words/data_organize/adjetivos/descição_de_arte/artistic')},
 {'word': 'brief',
  'path': WindowsPath('database/extract_data_video/data/extracted_data/words/data_organize/adjetivos/descição_de_arte/brief')},
 {'word': 'classical',
  'path': WindowsPath('database/extract_data_video/data/extracted_d